[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/36.%20The%20Berth%20%26%20Quay%20Length%20Design%20Problem/P37-Tier-2_executed.ipynb)

---

# 37. The Equipment Selection Problem (RTG vs. RMG vs. Straddle)
## Tier 2: Constraint Propagation with Backtracking

### Goal
Model the selection of container-handling equipment as a constraint satisfaction problem (CSP), then search for feasible equipment quantities using forward checking and backtracking. The search uses a fail-first variable-ordering heuristic and a least-constraining-value ordering heuristic. It can return the first feasible assignment or enumerate a bounded number of feasible assignments for descriptive comparison.

### Problem inputs and outputs
The inputs are the available equipment types, their costs and operating characteristics, the capacity and budget requirements for each block, an overall investment budget, and adjacency compatibility requirements. The search returns assignments of nonnegative integer equipment quantities to block-equipment pairs. Feasibility is determined by the stated budget, capacity, and adjacency constraints; the evaluation metrics describe feasible assignments but do not define an optimization objective.

### Model assumptions and scope
- Each block-equipment pair is a decision variable representing a count of equipment units. Equipment units are indivisible, so counts are nonnegative integers.
- Equipment productivity is specified in containers per hour, while block capacity requirements are specified in moves per day. The model assumes continuous operation for 24 hours per day and converts hourly productivity to daily capacity using that factor.
- Capital costs and budgets are measured in millions of dollars. Operating cost, maintenance, footprint, and energy consumption are recorded but are not constraints or ranking criteria in this CSP.
- A nonempty `compatibility_restrictions` list identifies equipment types prohibited in that block. Adjacent blocks must use at least one common equipment type; sharing a type means that its assigned count is positive in both blocks.
- This is a finite, feasibility-focused CSP. The first assignment found is not guaranteed to be cheapest or best under any evaluation metric. Enumeration is limited by `max_solutions`, so rankings apply only to the assignments returned by that bounded search.

### CSP formulation
A CSP consists of decision variables, a finite domain of candidate values for each variable, and constraints that rule out incompatible combinations. Here, for block \(b\) and equipment type \(e\), let \(x_{b,e}\) be the integer number of units assigned. Its initial domain is

$$
D_{b,e} = \{0,1,\ldots,\lfloor B_b/c_e \rfloor\},
$$

when type \(e\) is allowed in block \(b\); otherwise that variable is omitted. Here, \(B_b\) is the block's maximum capital budget and \(c_e\) is the capital cost per unit. The floor function gives the largest whole number of units that can fit within the block budget if no other equipment is purchased.

Every complete assignment must satisfy the following constraints:

$$
\begin{aligned}
\sum_e c_e x_{b,e} &\le B_b && \text{for each block } b,\\
24\sum_e p_e x_{b,e} &\ge R_b && \text{for each block } b,\\
\sum_b\sum_e c_e x_{b,e} &\le B_{\mathrm{total}},\\
\sum_e \mathbf{1}[x_{b,e}>0\ \text{and}\ x_{b+1,e}>0] &\ge 1 && \text{for each adjacent pair } (b,b+1).
\end{aligned}
$$

Here \(p_e\) is hourly productivity, \(R_b\) is the daily capacity requirement, \(B_{\mathrm{total}}\) is the overall budget, and \(\mathbf{1}[\cdot]\) equals 1 when its condition is true and 0 otherwise. The factor 24 expresses the continuous-operation assumption. In the code, the decision variables are named `block_<id>_<equipment>`, and these constraints are represented by `Constraint` objects.

### How propagation and backtracking work
A search state consists of the assignments made so far and the remaining domains. Forward checking tests candidate values against every constraint involving the affected variable. It removes a candidate when no completion of that constraint is possible using the current domains. Propagation repeats until no more values can be removed or a domain becomes empty. An empty domain proves that the current branch cannot lead to a feasible complete assignment.

Backtracking explores the remaining choices recursively. If a branch becomes infeasible, it restores the saved domains and tries another value. The fail-first (minimum-remaining-values) heuristic chooses an unassigned variable with the smallest current domain, aiming to expose a dead end early. The least-constraining-value heuristic tries values that preserve the greatest number of compatible choices for related variables.

Formally, if \(A\) is the current partial assignment and \(D\) denotes the current domains, the search returns a complete assignment satisfying every constraint, or reports that none exists. Propagation removes values that cannot participate in a feasible extension; recursion explores the values that remain. With \(n\) variables and at most \(d\) values per domain, an unpruned search can require \(O(d^n)\) assignments in the worst case. The heuristics can reduce practical search, but they do not change that worst-case bound.

The evaluation step reports cost, hourly and daily productivity, unit count, average flexibility, and productivity per million dollars. These are descriptive criteria, not a scalar objective optimized by the CSP.

In [ ]:
import time
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)

print('Equipment Selection: Constraint Propagation with Backtracking')
print('Finite-domain search with forward checking')

### Equipment and constraint data structures
The equipment record stores the supplied operational characteristics. The block record stores its capacity requirement, local budget, and any prohibited equipment types. A constraint associates the decision variables it involves with a function that tests whether the current assignments and remaining domains can still satisfy it.

In [ ]:
@dataclass
class EquipmentType:
    name: str
    capital_cost: float
    operating_cost: float
    productivity: float
    flexibility: float
    maintenance: float
    footprint: float
    energy_consumption: float


@dataclass
class BlockRequirement:
    block_id: int
    min_capacity: float
    max_budget: float
    compatibility_restrictions: List[str]


@dataclass
class Constraint:
    name: str
    variables: List[str]
    check_function: Callable[[Dict[str, int], Dict[str, List[int]]], bool]


equipment_types = {
    'RTG': EquipmentType(
        name='Rubber-Tired Gantry',
        capital_cost=2.5,
        operating_cost=0.8,
        productivity=25,
        flexibility=0.8,
        maintenance=0.3,
        footprint=200,
        energy_consumption=3.2,
    ),
    'RMG': EquipmentType(
        name='Rail-Mounted Gantry',
        capital_cost=3.2,
        operating_cost=0.6,
        productivity=35,
        flexibility=0.4,
        maintenance=0.25,
        footprint=150,
        energy_consumption=2.8,
    ),
    'Straddle': EquipmentType(
        name='Straddle Carrier',
        capital_cost=4.1,
        operating_cost=1.1,
        productivity=20,
        flexibility=0.9,
        maintenance=0.4,
        footprint=100,
        energy_consumption=4.5,
    ),
}

print('Equipment characteristics:')
for equipment_code, equipment in equipment_types.items():
    print(f'{equipment_code}: {equipment.name}')
    print(f'  Capital cost: ${equipment.capital_cost:.1f}M per unit')
    print(f'  Productivity: {equipment.productivity} containers/hour per unit')
    print(f'  Flexibility score: {equipment.flexibility:.1f}')
    print()

### Constraint propagation system
The implementation stores one finite domain for every allowed block-equipment pair. Constraint checks are designed to be safe on partial assignments: they reject a branch only when the assigned values and remaining domains cannot satisfy the relevant constraint. This is what allows forward checking to prune values without discarding feasible complete assignments.

In [ ]:
class ConstraintPropagationSystem:
    def __init__(
        self,
        equipment_types: Dict[str, EquipmentType],
        block_requirements: List[BlockRequirement],
        total_budget: float = 180.0,
        operating_hours_per_day: float = 24.0,
    ):
        self.equipment_types = equipment_types
        self.block_requirements = block_requirements
        self.total_budget = total_budget
        self.operating_hours_per_day = operating_hours_per_day
        self.domains: Dict[str, List[int]] = {}
        self.initial_domains: Dict[str, List[int]] = {}
        self.constraints: List[Constraint] = []
        self.solutions: List[Dict[str, int]] = []
        self.nodes_explored = 0
        self.constraints_checked = 0
        self.pruned_values = 0
        self.backtracks = 0

        self._initialize_domains()
        self.initial_domains = {
            variable: domain.copy()
            for variable, domain in self.domains.items()
        }
        self._define_constraints()

    def _initialize_domains(self) -> None:
        for block in self.block_requirements:
            for equipment_code, equipment in self.equipment_types.items():
                if equipment_code in block.compatibility_restrictions:
                    continue
                max_units = int(block.max_budget / equipment.capital_cost)
                variable = f'block_{block.block_id}_{equipment_code}'
                self.domains[variable] = list(range(max_units + 1))

    def _define_constraints(self) -> None:
        constraints = []

        for block in self.block_requirements:
            block_variables = [
                f'block_{block.block_id}_{equipment_code}'
                for equipment_code in self.equipment_types
                if f'block_{block.block_id}_{equipment_code}' in self.domains
            ]

            def block_budget_check(assignment, domains, block=block,
                                   variables=block_variables):
                assigned_cost = sum(
                    assignment[variable]
                    * self.equipment_types[variable.split('_', 2)[2]].capital_cost
                    for variable in variables
                    if variable in assignment
                )
                return assigned_cost <= block.max_budget

            constraints.append(
                Constraint(
                    name=f'block_budget_{block.block_id}',
                    variables=block_variables,
                    check_function=block_budget_check,
                )
            )

            def block_capacity_check(assignment, domains, block=block,
                                     variables=block_variables):
                possible_daily_capacity = 0.0
                for variable in variables:
                    equipment_code = variable.split('_', 2)[2]
                    equipment = self.equipment_types[equipment_code]
                    if variable in assignment:
                        units = assignment[variable]
                    else:
                        units = max(domains[variable], default=-1)
                    possible_daily_capacity += (
                        units
                        * equipment.productivity
                        * self.operating_hours_per_day
                    )
                return possible_daily_capacity >= block.min_capacity

            constraints.append(
                Constraint(
                    name=f'block_capacity_{block.block_id}',
                    variables=block_variables,
                    check_function=block_capacity_check,
                )
            )

        all_variables = list(self.domains)

        def total_budget_check(assignment, domains):
            assigned_cost = sum(
                assignment[variable]
                * self.equipment_types[variable.split('_', 2)[2]].capital_cost
                for variable in all_variables
                if variable in assignment
            )
            return assigned_cost <= self.total_budget

        constraints.append(
            Constraint(
                name='total_budget',
                variables=all_variables,
                check_function=total_budget_check,
            )
        )

        for first_block, second_block in zip(
            self.block_requirements, self.block_requirements[1:]
        ):
            first_id = first_block.block_id
            second_id = second_block.block_id
            adjacent_variables = [
                variable
                for equipment_code in self.equipment_types
                for variable in (
                    f'block_{first_id}_{equipment_code}',
                    f'block_{second_id}_{equipment_code}',
                )
                if variable in self.domains
            ]

            def adjacency_check(assignment, domains, first_id=first_id,
                                second_id=second_id):
                for equipment_code in self.equipment_types:
                    first_variable = f'block_{first_id}_{equipment_code}'
                    second_variable = f'block_{second_id}_{equipment_code}'
                    if first_variable not in self.domains or second_variable not in self.domains:
                        continue
                    first_can_be_positive = (
                        assignment[first_variable] > 0
                        if first_variable in assignment
                        else any(value > 0 for value in domains[first_variable])
                    )
                    second_can_be_positive = (
                        assignment[second_variable] > 0
                        if second_variable in assignment
                        else any(value > 0 for value in domains[second_variable])
                    )
                    if first_can_be_positive and second_can_be_positive:
                        return True
                return False

            constraints.append(
                Constraint(
                    name=f'adjacency_{first_id}_{second_id}',
                    variables=adjacent_variables,
                    check_function=adjacency_check,
                )
            )

        self.constraints = constraints

    def _constraint_is_possible(
        self,
        constraint: Constraint,
        assignment: Dict[str, int],
        domains: Dict[str, List[int]],
    ) -> bool:
        self.constraints_checked += 1
        return constraint.check_function(assignment, domains)

    def forward_checking(
        self, variable: str, assigned_values: Dict[str, int]
    ) -> bool:
        """Prune values that cannot satisfy a constraint in this branch."""
        changed = True
        while changed:
            changed = False
            for constraint in self.constraints:
                if variable not in constraint.variables and not any(
                    item in assigned_values for item in constraint.variables
                ):
                    continue

                for candidate_variable in constraint.variables:
                    if candidate_variable in assigned_values:
                        continue

                    values_to_remove = []
                    for value in self.domains[candidate_variable]:
                        trial_assignment = assigned_values.copy()
                        trial_assignment[candidate_variable] = value
                        if not self._constraint_is_possible(
                            constraint, trial_assignment, self.domains
                        ):
                            values_to_remove.append(value)

                    if values_to_remove:
                        self.domains[candidate_variable] = [
                            value
                            for value in self.domains[candidate_variable]
                            if value not in values_to_remove
                        ]
                        self.pruned_values += len(values_to_remove)
                        changed = True
                        if not self.domains[candidate_variable]:
                            return False

            for constraint in self.constraints:
                if not self._constraint_is_possible(
                    constraint, assigned_values, self.domains
                ):
                    return False
        return True

    def select_unassigned_variable(
        self, assignment: Dict[str, int]
    ) -> Optional[str]:
        unassigned = [
            variable for variable in self.domains if variable not in assignment
        ]
        if not unassigned:
            return None
        return min(unassigned, key=lambda variable: len(self.domains[variable]))

    def order_domain_values(self, variable: str) -> List[int]:
        """Try values that preserve more choices for related variables first."""
        related_constraints = [
            constraint
            for constraint in self.constraints
            if variable in constraint.variables
        ]
        unassigned_neighbors = {
            neighbor
            for constraint in related_constraints
            for neighbor in constraint.variables
            if neighbor != variable
        }
        unassigned_neighbors = {
            neighbor
            for neighbor in unassigned_neighbors
            if neighbor in self.domains
        }

        def remaining_support(value: int) -> int:
            trial_assignment = {variable: value}
            score = 0
            for constraint in related_constraints:
                for neighbor in unassigned_neighbors.intersection(
                    constraint.variables
                ):
                    for neighbor_value in self.domains[neighbor]:
                        trial = trial_assignment.copy()
                        trial[neighbor] = neighbor_value
                        if self._constraint_is_possible(
                            constraint, trial, self.domains
                        ):
                            score += 1
            return score

        return sorted(
            self.domains[variable],
            key=lambda value: (-remaining_support(value), value),
        )

    def is_assignment_complete(self, assignment: Dict[str, int]) -> bool:
        return len(assignment) == len(self.domains)

    def is_consistent(self, assignment: Dict[str, int]) -> bool:
        return all(
            self._constraint_is_possible(constraint, assignment, self.domains)
            for constraint in self.constraints
        )

    def _reset_search_state(self) -> None:
        self.domains = {
            variable: domain.copy()
            for variable, domain in self.initial_domains.items()
        }
        self.nodes_explored = 0
        self.constraints_checked = 0
        self.pruned_values = 0
        self.backtracks = 0

    def backtrack_search(
        self, assignment: Optional[Dict[str, int]] = None
    ) -> Optional[Dict[str, int]]:
        """Return the first feasible complete assignment, if one is found."""
        if assignment is None:
            self._reset_search_state()
            assignment = {}

        self.nodes_explored += 1
        if self.is_assignment_complete(assignment):
            return assignment.copy() if self.is_consistent(assignment) else None

        variable = self.select_unassigned_variable(assignment)
        if variable is None:
            return None

        for value in self.order_domain_values(variable):
            saved_domains = {
                item: domain.copy() for item, domain in self.domains.items()
            }
            assignment[variable] = value

            if self.forward_checking(variable, assignment):
                result = self.backtrack_search(assignment)
                if result is not None:
                    return result

            self.domains = saved_domains
            del assignment[variable]
            self.backtracks += 1

        return None

    def find_all_solutions(self, max_solutions: int = 100) -> List[Dict[str, int]]:
        """Enumerate up to max_solutions feasible assignments."""
        self._reset_search_state()
        solutions: List[Dict[str, int]] = []

        def search(assignment: Dict[str, int]) -> None:
            if len(solutions) >= max_solutions:
                return
            self.nodes_explored += 1

            if self.is_assignment_complete(assignment):
                if self.is_consistent(assignment):
                    solutions.append(assignment.copy())
                return

            variable = self.select_unassigned_variable(assignment)
            if variable is None:
                return

            for value in self.order_domain_values(variable):
                saved_domains = {
                    item: domain.copy() for item, domain in self.domains.items()
                }
                assignment[variable] = value

                if self.forward_checking(variable, assignment):
                    search(assignment)

                self.domains = saved_domains
                del assignment[variable]
                self.backtracks += 1

                if len(solutions) >= max_solutions:
                    return

        search({})
        self.solutions = solutions
        return solutions

    def evaluate_solution(self, solution: Dict[str, int]) -> Dict[str, object]:
        total_cost = 0.0
        total_hourly_productivity = 0.0
        total_units = 0
        weighted_flexibility = 0.0

        for variable, units in solution.items():
            equipment_code = variable.split('_', 2)[2]
            equipment = self.equipment_types[equipment_code]
            total_cost += units * equipment.capital_cost
            total_hourly_productivity += units * equipment.productivity
            total_units += units
            weighted_flexibility += units * equipment.flexibility

        average_flexibility = (
            weighted_flexibility / total_units if total_units else 0.0
        )
        daily_capacity = (
            total_hourly_productivity * self.operating_hours_per_day
        )

        return {
            'solution': solution,
            'total_cost': total_cost,
            'total_hourly_productivity': total_hourly_productivity,
            'daily_capacity': daily_capacity,
            'total_units': total_units,
            'avg_flexibility': average_flexibility,
            'cost_efficiency': (
                total_hourly_productivity / total_cost if total_cost else 0.0
            ),
        }

    def solution_satisfies_constraints(self, solution: Dict[str, int]) -> bool:
        if set(solution) != set(self.domains):
            return False
        return all(
            constraint.check_function(solution, self.initial_domains)
            for constraint in self.constraints
        )


print('Constraint propagation system is ready.')

### Set up the instance and run the search
Each block-equipment pair receives a domain based on the block's local budget. The solver then searches for one complete feasible assignment. Search counters describe the work performed by this first-solution search; pruning counts are cumulative branch-level removals and can include repeated removals in different branches.

In [ ]:
block_requirements = [
    BlockRequirement(0, 1800, 45, []),
    BlockRequirement(1, 2000, 50, []),
    BlockRequirement(2, 1900, 48, []),
    BlockRequirement(3, 2100, 52, []),
    BlockRequirement(4, 1950, 47, []),
    BlockRequirement(5, 2050, 51, []),
    BlockRequirement(6, 1850, 46, []),
    BlockRequirement(7, 2200, 55, []),
]

cp_system = ConstraintPropagationSystem(
    equipment_types=equipment_types,
    block_requirements=block_requirements,
    total_budget=180.0,
    operating_hours_per_day=24.0,
)

print('=== PROBLEM SETUP ===\n')
print(f'Number of blocks: {len(block_requirements)}')
print(f'Total investment budget: ${cp_system.total_budget:.1f}M')
print(f'Equipment types: {len(equipment_types)}')
print(f'Decision variables: {len(cp_system.domains)}')
print(f'Constraints: {len(cp_system.constraints)}')
print(f'Operating hours assumed per day: {cp_system.operating_hours_per_day:.0f}\n')

print('Block requirements and initial domain sizes:')
for block in block_requirements:
    block_variables = [
        variable
        for variable in cp_system.domains
        if variable.startswith(f'block_{block.block_id}_')
    ]
    domain_size = sum(len(cp_system.domains[variable]) for variable in block_variables)
    print(
        f'  Block {block.block_id}: capacity at least {block.min_capacity} moves/day, '
        f'budget at most ${block.max_budget}M, {domain_size} initial domain values'
    )

print('\n=== FIRST-FEASIBLE-ASSIGNMENT SEARCH ===\n')
start_time = time.perf_counter()
first_solution = cp_system.backtrack_search()
search_time = time.perf_counter() - start_time

first_search_stats = {
    'nodes_explored': cp_system.nodes_explored,
    'constraints_checked': cp_system.constraints_checked,
    'pruned_values': cp_system.pruned_values,
    'backtracks': cp_system.backtracks,
}

print(f'Search time: {search_time:.6f} seconds')
print(f'Nodes explored: {first_search_stats["nodes_explored"]}')
print(f'Constraint checks: {first_search_stats["constraints_checked"]}')
print(f'Domain values pruned: {first_search_stats["pruned_values"]}')
print(f'Backtracks: {first_search_stats["backtracks"]}')

if first_solution is not None:
    first_evaluation = cp_system.evaluate_solution(first_solution)
    print('\nFirst feasible equipment assignment:')
    for block in block_requirements:
        selected = [
            (equipment_code, first_solution[f'block_{block.block_id}_{equipment_code}'])
            for equipment_code in equipment_types
            if f'block_{block.block_id}_{equipment_code}' in first_solution
            and first_solution[f'block_{block.block_id}_{equipment_code}'] > 0
        ]
        description = ', '.join(
            f'{units} {equipment_code}' for equipment_code, units in selected
        ) or 'no units'
        print(f'  Block {block.block_id}: {description}')

    print('\nAssignment metrics:')
    print(f'  Capital cost: ${first_evaluation["total_cost"]:.1f}M')
    print(
        '  Hourly productivity: '
        f'{first_evaluation["total_hourly_productivity"]:.0f} containers/hour'
    )
    print(f'  Daily capacity under the stated assumption: {first_evaluation["daily_capacity"]:.0f} moves/day')
    print(f'  Equipment units: {first_evaluation["total_units"]}')
    print(f'  Unit-weighted flexibility: {first_evaluation["avg_flexibility"]:.3f}')
    print(f'  Productivity per $M: {first_evaluation["cost_efficiency"]:.2f} containers/hour per $M')
else:
    print('No feasible assignment exists for the stated model and data.')

### Enumerate and compare feasible assignments
The enumeration routine uses the same domains, constraints, propagation, and backtracking method. It stops after the requested limit, so the returned list is a search-ordered sample when more assignments exist. Sorting this list by a metric describes the returned assignments; it does not establish a global optimum unless the complete feasible set has been enumerated.

In [ ]:
print('=== BOUNDED FEASIBLE-ASSIGNMENT ENUMERATION ===\n')
start_time = time.perf_counter()
all_solutions = cp_system.find_all_solutions(max_solutions=50)
multi_solution_time = time.perf_counter() - start_time
evaluated_solutions = [
    cp_system.evaluate_solution(solution) for solution in all_solutions
]

print(f'Assignments returned: {len(all_solutions)}')
print(f'Enumeration time: {multi_solution_time:.6f} seconds')
print('Enumeration limit: 50 assignments')

if evaluated_solutions:
    best_cost = min(evaluated_solutions, key=lambda item: item['total_cost'])
    best_productivity = max(
        evaluated_solutions,
        key=lambda item: item['total_hourly_productivity'],
    )
    best_efficiency = max(
        evaluated_solutions,
        key=lambda item: item['cost_efficiency'],
    )
    best_flexibility = max(
        evaluated_solutions,
        key=lambda item: item['avg_flexibility'],
    )

    print('\nBest metric values among returned assignments:')
    for label, evaluation in (
        ('Lowest capital cost', best_cost),
        ('Highest hourly productivity', best_productivity),
        ('Highest productivity per $M', best_efficiency),
        ('Highest average flexibility', best_flexibility),
    ):
        print(
            f'  {label}: cost ${evaluation["total_cost"]:.1f}M, '
            f'productivity {evaluation["total_hourly_productivity"]:.0f} containers/hour, '
            f'flexibility {evaluation["avg_flexibility"]:.3f}'
        )

    comparison_df = pd.DataFrame(
        [
            {
                'Assignment': index + 1,
                'Capital cost ($M)': evaluation['total_cost'],
                'Productivity (containers/hour)': evaluation['total_hourly_productivity'],
                'Daily capacity (moves/day)': evaluation['daily_capacity'],
                'Equipment units': evaluation['total_units'],
                'Average flexibility': evaluation['avg_flexibility'],
                'Productivity per $M': evaluation['cost_efficiency'],
            }
            for index, evaluation in enumerate(evaluated_solutions[:10])
        ]
    )
    print('\nFirst returned assignments:')
    print(comparison_df.round(2).to_string(index=False))
else:
    print('The bounded enumeration returned no feasible assignments.')

### Search performance and assignment visualization
The plots summarize the assignments returned by bounded enumeration and the work recorded during the first-solution search. Cost and productivity are separate measures: the scatter plot shows their observed trade-off in the returned set, not a complete Pareto frontier unless all feasible assignments have been enumerated.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('Constraint Propagation and Backtracking Results', fontsize=15)
ax1, ax2, ax3, ax4 = axes.ravel()

if evaluated_solutions:
    costs = [item['total_cost'] for item in evaluated_solutions]
    productivities = [
        item['total_hourly_productivity'] for item in evaluated_solutions
    ]
    flexibilities = [item['avg_flexibility'] for item in evaluated_solutions]
    efficiencies = [item['cost_efficiency'] for item in evaluated_solutions]

    ax1.scatter(costs, productivities, alpha=0.7, label='Returned assignments')
    ax1.scatter(
        best_cost['total_cost'],
        best_cost['total_hourly_productivity'],
        color='red',
        marker='*',
        s=140,
        label='Lowest cost in returned set',
    )
    ax1.set_xlabel('Capital cost ($M)')
    ax1.set_ylabel('Productivity (containers/hour)')
    ax1.set_title('Capital Cost and Hourly Productivity')
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.hist(flexibilities, bins=15, color='skyblue', edgecolor='black')
    ax2.axvline(
        best_flexibility['avg_flexibility'],
        color='red',
        linestyle='--',
        label='Highest in returned set',
    )
    ax2.set_xlabel('Unit-weighted average flexibility score')
    ax2.set_ylabel('Number of assignments')
    ax2.set_title('Flexibility Scores')
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    metric_names = ['Nodes', 'Checks', 'Pruned values', 'Backtracks']
    metric_values = [
        first_search_stats['nodes_explored'],
        first_search_stats['constraints_checked'],
        first_search_stats['pruned_values'],
        first_search_stats['backtracks'],
    ]
    bars = ax3.bar(metric_names, metric_values, color='steelblue', alpha=0.8)
    ax3.set_ylabel('Count')
    ax3.set_title('First-Solution Search Statistics')
    ax3.tick_params(axis='x', rotation=20)
    ax3.grid(True, axis='y', alpha=0.3)
    for bar, value in zip(bars, metric_values):
        ax3.annotate(
            f'{value:,}',
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords='offset points',
            ha='center',
            va='bottom',
        )

    indices = range(len(efficiencies))
    ax4.plot(indices, efficiencies, marker='o', linewidth=1.5)
    ax4.axhline(
        best_efficiency['cost_efficiency'],
        color='red',
        linestyle='--',
        label='Highest in returned set',
    )
    ax4.set_xlabel('Returned assignment index')
    ax4.set_ylabel('Productivity per $M (containers/hour)')
    ax4.set_title('Productivity per Capital Cost')
    ax4.grid(True, alpha=0.3)
    ax4.legend()
else:
    for axis in axes.ravel():
        axis.text(
            0.5,
            0.5,
            'No feasible assignments to display',
            ha='center',
            va='center',
            transform=axis.transAxes,
        )
        axis.set_axis_off()

fig.tight_layout()
plt.show()

### Validate the returned assignments
A search result is useful only if it satisfies the model. The following checks verify that each returned assignment contains every decision variable and satisfies all block budgets, block capacity requirements, the total budget, and adjacent-block compatibility. These checks validate the CSP results; they do not introduce another search method.

In [ ]:
if all_solutions:
    validation_results = [
        cp_system.solution_satisfies_constraints(solution)
        for solution in all_solutions
    ]
    assert all(validation_results), 'At least one returned assignment violates a constraint.'

    for solution in all_solutions:
        evaluation = cp_system.evaluate_solution(solution)
        assert evaluation['total_cost'] <= cp_system.total_budget
        for block in block_requirements:
            block_cost = 0.0
            block_daily_capacity = 0.0
            for equipment_code, equipment in equipment_types.items():
                variable = f'block_{block.block_id}_{equipment_code}'
                if variable in solution:
                    units = solution[variable]
                    block_cost += units * equipment.capital_cost
                    block_daily_capacity += (
                        units
                        * equipment.productivity
                        * cp_system.operating_hours_per_day
                    )
            assert block_cost <= block.max_budget
            assert block_daily_capacity >= block.min_capacity

    print(
        f'Validated all {len(all_solutions)} returned assignments against '
        'the complete constraint set.'
    )
else:
    print('No returned assignments are available for validation.')

### Results summary and interpretation
The summary below reports the search configuration, first-solution search statistics, and the number of assignments returned by bounded enumeration. A feasible assignment demonstrates that at least one complete configuration satisfies the encoded model. No assignment indicates either infeasibility under the stated assumptions or a need to inspect the model inputs and constraints; it is not evidence that a different algorithm would be preferable.

The search is complete for the finite domains when run without an enumeration cutoff: backtracking explores the remaining values unless a sound constraint check proves a branch infeasible. With a cutoff, the enumeration reports only the assignments encountered before reaching that limit. The heuristic ordering affects discovery order, not the definition of feasibility.

In [ ]:
print('=== CONSTRAINT PROPAGATION WITH BACKTRACKING: SUMMARY ===\n')
print('Problem: Container-terminal equipment selection')
print('Method: Finite-domain CSP with forward checking and backtracking\n')

print('MODEL CONFIGURATION:')
print(f'  Blocks: {len(block_requirements)}')
print(f'  Equipment types: {len(equipment_types)}')
print(f'  Decision variables: {len(cp_system.domains)}')
print(f'  Constraints: {len(cp_system.constraints)}')
print(f'  Total budget: ${cp_system.total_budget:.1f}M')
print(f'  Operating-hours assumption: {cp_system.operating_hours_per_day:.0f} hours/day\n')

print('FIRST-SOLUTION SEARCH:')
print(f'  Nodes explored: {first_search_stats["nodes_explored"]:,}')
print(f'  Constraint checks: {first_search_stats["constraints_checked"]:,}')
print(f'  Domain values pruned: {first_search_stats["pruned_values"]:,}')
print(f'  Backtracks: {first_search_stats["backtracks"]:,}')
print(f'  Elapsed time: {search_time:.6f} seconds')

print('\nFEASIBILITY AND ENUMERATION:')
if first_solution is None:
    print('  No feasible assignment was found.')
else:
    print('  A feasible complete assignment was found.')
    print(f'  Assignments returned by bounded enumeration: {len(all_solutions)}')
    if evaluated_solutions:
        print(
            '  Lowest capital cost among returned assignments: '
            f'${best_cost["total_cost"]:.1f}M'
        )
        print(
            '  Highest hourly productivity among returned assignments: '
            f'{best_productivity["total_hourly_productivity"]:.0f} containers/hour'
        )
        print(
            '  Highest average flexibility among returned assignments: '
            f'{best_flexibility["avg_flexibility"]:.3f}'
        )

print('\nINTERPRETATION:')
print('  Forward checking removes values that cannot satisfy their constraints.')
print('  Backtracking restores domains when a branch cannot be completed.')
print('  The first feasible assignment is not an optimized solution.')
print('  Metric rankings apply only to the assignments returned by the bounded search.')